# Practical 14 — Spell Checking

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To build an edit-distance spelling corrector from scratch, measure how accurate it is on synthetic typos, and test what happens when it is run on the review corpus *after* Practical 1's cleaning — including what its "corrections" do to words that carry meaning.

## Theory

A spell checker has two jobs: **detect** a word that is probably wrong, then **suggest** the word the writer most likely meant.

This practical builds the classic edit-distance approach:
- **Detect:** a word is flagged if it is not in a vocabulary of known words.
- **Generate candidates:** every string one *edit* away from the flagged word, where an edit is a **deletion**, **transposition** (swap two adjacent letters), **replacement**, or **insertion** of one letter. If nothing one edit away is a real word, try two edits.
- **Choose:** among the real-word candidates, pick the **most frequent** one in a large reference corpus. Frequency stands in for "which word is the writer most likely to have meant".

The vocabulary and frequencies come from NLTK's `movie_reviews` corpus (about 1.6 million words), so they match our domain. A `min_count` setting drops rare words: a big real-world corpus contains typos of its own, and a misspelling that appears once would otherwise count as a "real" word.

Two limits are built into this approach, and this practical tests both:
- It only asks whether a word **exists**. A typo that happens to spell a *different* real word ("sea" for "see") is invisible to it.
- It judges **one word at a time** with no sentence context.

There is also a link back to the earlier practicals. Practical 1's `clean_text()` strips apostrophes, so "can't" becomes "cant" before any later step sees it. A spell checker has no way to know that "cant" was once a contraction. This practical checks what it does with words like that.

## Algorithm

1. Build word-frequency tables from the movie review corpus, with and without a minimum-count filter.
2. Correct a sentence containing genuine typos and check the results.
3. Run the spell checker over our cleaned review tokens, list every word it flags, and look at each proposed correction.
4. Test a sentence made entirely of valid words that are used wrongly, and see whether anything is flagged.
5. Measure accuracy on synthetic typos: one edit away vs two edits away, correcting up to distance 1 vs distance 2, and note how long each takes.
6. Repeat the accuracy test with five different random seeds and look at the spread, to see how stable the number is.

In [ ]:
import sys, os, time
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

nltk.download("movie_reviews")
from nltk.corpus import movie_reviews

import preprocessing
import tokenizer
import spellcheck as sc

all_words = list(movie_reviews.words())
print(f"Reference corpus: {len(all_words):,} words")

df = pd.read_csv("../datasets/sample_reviews.csv")
tokens_cleaned = df["review"].apply(
    lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r))
).tolist()


### Step 1 — Vocabulary size with different minimum counts

In [ ]:
counts_all = sc.build_word_counts(all_words, min_count=1)
counts_min5 = sc.build_word_counts(all_words, min_count=5)

print(f"Vocabulary, min_count=1: {len(counts_all):,} words")
print(f"Vocabulary, min_count=5: {len(counts_min5):,} words")
print()
print("Most common words (min_count=5):", counts_min5.most_common(8))


### Step 2 — Correcting genuine typos

In [ ]:
typo_sentence = "this movei was fantastc and the actng was terible".split()

for word in typo_sentence:
    fixed = sc.correct(word, counts_min5)
    marker = "  <-- changed" if fixed != word else ""
    print(f"{word:<10} -> {fixed}{marker}")


### Step 3 — Run it on our cleaned review tokens

Our reviews contain no deliberate typos, so anything flagged here is a word the vocabulary doesn't recognise for some other reason. Look at both settings: `min_count=1` (every corpus word counts as real) and `min_count=5` (rare words, including the corpus's own typos, are dropped).

**For each flagged word, ask two things: why was it flagged, and does the proposed "correction" preserve what the reviewer meant? Pay particular attention to any word that carried negation in the original review.**

In [ ]:
for label, counts in [("min_count=1", counts_all), ("min_count=5", counts_min5)]:
    unknown = sc.find_unknown_words(tokens_cleaned, counts)
    print(f"--- {label}: {len(unknown)} distinct flagged words ---")
    for word, freq in unknown.most_common():
        print(f"  {word!r:<10} (x{freq})  ->  {sc.correct(word, counts)!r}")
    print()


To see where a flagged word came from, find the original reviews it appears in:

In [ ]:
# Change this list to any flagged words you want to trace back to the raw text.
words_to_trace = ["cant", "wont", "dont"]

for target in words_to_trace:
    for i, tokens in enumerate(tokens_cleaned):
        if target in tokens:
            print(f"{target!r} appears in review {df.loc[i, 'id']}: {df.loc[i, 'review']}")


### Step 4 — Typos that are real words

In [ ]:
real_word_errors = "i went to sea this film and their was no plot".split()

flagged = [w for w in real_word_errors if w not in counts_min5]
corrected = " ".join(sc.correct(w, counts_min5) for w in real_word_errors)

print("Input:    ", " ".join(real_word_errors))
print("Flagged:  ", flagged)
print("Corrected:", corrected)


### Step 5 — Accuracy on synthetic typos (and what it costs)

In [ ]:
# 500 common words of length 4+, each corrupted with random typos, then corrected.
test_words = [w for w, c in counts_min5.most_common(2000) if len(w) >= 4][:500]
print(f"Testing on {len(test_words)} words")
print()

for n_edits in (1, 2):
    for max_distance in (1, 2):
        start = time.time()
        result = sc.evaluate_corrector(test_words, counts_min5,
                                       max_distance=max_distance, n_edits=n_edits)
        elapsed = time.time() - start
        print(f"typos {n_edits} edit(s) away | corrector max_distance={max_distance} | "
              f"accuracy {result['accuracy']:.1%} ({result['recovered']}/{result['tested']}) | {elapsed:.1f}s")


**The last row can take 10-20 seconds. That is part of the result: note which settings are slow and which are instant, and think about why.**

### Step 6 — How stable is the accuracy number?

In [ ]:
accuracies = []
for seed in range(5):
    result = sc.evaluate_corrector(test_words, counts_min5, max_distance=1, n_edits=1, seed=seed)
    accuracies.append(result["accuracy"])

print("Accuracy across 5 different random seeds:", [f"{a:.1%}" for a in accuracies])
print(f"Spread (max - min): {max(accuracies) - min(accuracies):.1%}")


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the vocabulary sizes, the typo-sentence corrections, the flagged words and their proposed corrections (both min_count settings), what the real-word-error test flagged and corrected, the four accuracy lines with timings, and the five-seed spread. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did the corrector fix the genuine typos in Step 2? Was every correction the word you would have chosen?
- In Step 3, which words were flagged under each `min_count` setting, and why were they flagged (none of them is a real typo)? Why did raising `min_count` flag more words?
- For the flagged words, which proposed corrections preserved the reviewer's meaning, and which changed or reversed it? Trace at least one back to its original review.
- In Step 4, was anything flagged? What does that show about the kind of error this approach can and cannot catch?
- In Step 5, how did accuracy change between one-edit and two-edit typos, and what did allowing `max_distance=2` cost in time? Why is it so much slower?
- In Step 6, how much did accuracy move between seeds? Compare this with the spread you saw in Practical 11, and explain the difference.
- Given all of this, where in a pipeline should spell checking go relative to Practical 1's cleaning steps?

*(Write 5-7 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **What four kinds of single edit does an edit-distance spell corrector consider?**
   Deleting a letter, transposing two adjacent letters, replacing a letter, and inserting a letter.

2. **Why choose the most frequent candidate among the real words one edit away?**
   Frequency acts as a simple language model: a common word is more likely to be what the writer meant than a rare one that happens to be equally close in spelling.

3. **What is the difference between a non-word error and a real-word error, and which one can this corrector catch?**
   A non-word error produces a string that isn't a word at all ("movei"); a real-word error produces a valid word used wrongly ("sea" for "see"). This corrector only checks vocabulary membership, so it catches non-word errors and cannot see real-word errors. Catching those needs sentence context.

4. **Why does correcting up to two edits take so much longer than one?**
   Each word has a few hundred strings one edit away, and two edits means taking every one of those and editing it again, so the candidate set grows roughly with the square. Each candidate must then be looked up in the vocabulary.

5. **Why can running a spell checker after aggressive text cleaning be risky?**
   Cleaning steps such as stripping apostrophes turn contractions into strings that aren't real words. The spell checker flags them and replaces them with the nearest real word, which can be a different word entirely, including one that changes or removes a negation.

6. **Why did the accuracy number stay stable across random seeds here when it did not in Practical 11?**
   Here each run tests several hundred words, so one unlucky example barely moves the total. Practical 11 had only 4-5 test examples, where a single example shifts the result by 20-25 percentage points.
